# Adaptive Nervous System v0.3.0

## ทดลองการตัดสินใจภายใต้เวลาขนย้ายทรัพยากร

โน้ตบุ๊กนี้รัน validation จาก **Adaptive_Nervous_System_v030.zip** บน CPU ไม่ต้องใช้ GPU หรือ API key เปิดใน Google Colab ได้จากคอมพิวเตอร์หรือโทรศัพท์

**สถานะของตัวเลือกที่บรรจุในแพ็กเกจ:** workflow รันสำเร็จ แต่ **v3_full ยังไม่ผ่านเกณฑ์ noninferiority เทียบ v0.2 และเป้าหมายด้านความเสียหายกรณีแย่/งานค้างปลายช่วงบางเงื่อนไข** จึงยังไม่ใช่ตัวเลือกที่ผ่านเกณฑ์ให้แทนรุ่นเดิม โน้ตบุ๊กจะคำนวณผล PASS/FAIL ใหม่จากการรันของคุณตามกติกาเดิมโดยอัตโนมัติ

ข้อมูลทั้งหมดเป็น **การจำลองคิวสามขั้น** ไม่มีการเชื่อมต่อระบบบริการจริงหรือสั่งย้ายทรัพยากรภายนอก การทดลองแยก 3 เงื่อนไข: ย้ายทันที, ใช้เวลา 1 นาที และใช้เวลา 3 นาที แสดงผลของแต่ละเงื่อนไขแยกกัน

| วิธี | บทบาท |
| --- | --- |
| `static` | จัดสรรคงที่ 69 / 69 / 69 |
| `abc_capacity` | Gate ที่กรองข้อเสนอ ABC จาก v0.2 |
| `capacity_v2` | ตัวเลือกจากแบบจำลอง v0.2 |
| `capacity_balance` | วิธีฐานที่ปรับสมดุลตามอัตราการทำงานที่สังเกต |
| `v3_signal_only` | เพิ่มการตรวจสัญญาณคงอยู่/แยกกำลังการทำงาน โดยยังใช้การคาดการณ์เดิม |
| `v3_full` | เพิ่มสถานการณ์ความไม่แน่นอนรายขั้น, ตรวจงานค้างปลายช่วง และคิดเวลาขนย้ายในการคาดการณ์ |

การเทียบ full กับ signal_only เปลี่ยนหลายส่วนพร้อมกัน จึงไม่ใช่การแยกผลของสถานการณ์ความไม่แน่นอนเพียงส่วนเดียว

**วิธีใช้:** รันเซลล์ตามลำดับ และอัปโหลด ZIP รุ่น v0.3 เมื่อมีคำถาม การทดลองเต็มมี **352 สถานการณ์ × 3 เงื่อนไข × 8 rollout = 8,448 rollout** อาจใช้เวลาหลายนาทีตามเครื่อง

แพ็กเกจมี protocol ที่ตรึงแล้ว ไม่ต้องเรียก freeze ใหม่ หากตรวจ hash ไม่ผ่าน ให้ตรวจความครบถ้วนของแพ็กเกจก่อนทดลอง โค้ดโน้ตบุ๊กตรวจ syntax แล้ว แต่ส่วนติดต่อ Colab ยังไม่ได้ทดสอบโดยตรง

## 1. อัปโหลดแพ็กเกจ

เลือก **Adaptive_Nervous_System_v030.zip** เพียงไฟล์เดียว โค้ดตรวจเส้นทางก่อนแตก ZIP และค้นหา `run_v3.py` โดยอัตโนมัติ

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io
import stat
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("กรุณาเลือก Adaptive_Nervous_System_v030.zip เพียงไฟล์เดียว")
upload_name, upload_bytes = next(iter(uploaded.items()))
if not upload_name.lower().endswith(".zip"):
    raise ValueError("ต้องเลือกไฟล์ ZIP")
EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="ans_v030_")).resolve()
with zipfile.ZipFile(io.BytesIO(upload_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 10000 or sum(m.file_size for m in members) > 512 * 1024 * 1024:
        raise ValueError("ขนาดแพ็กเกจเกินขอบเขตที่คาดไว้")
    destinations = set()
    for member in members:
        relative = PurePosixPath(member.filename)
        target = (EXTRACT_ROOT / member.filename).resolve()
        if (relative.is_absolute() or ".." in relative.parts
                or "\\" in member.filename
                or not target.is_relative_to(EXTRACT_ROOT)
                or stat.S_ISLNK(member.external_attr >> 16)):
            raise ValueError(f"เส้นทาง ZIP ไม่ถูกต้อง: {member.filename!r}")
        if target in destinations:
            raise ValueError(f"เส้นทางซ้ำใน ZIP: {member.filename!r}")
        destinations.add(target)
    archive.extractall(EXTRACT_ROOT)
candidates = sorted(EXTRACT_ROOT.rglob("run_v3.py"))
if len(candidates) != 1:
    raise ValueError("ต้องพบโครงการ v0.3 เพียงหนึ่งชุด")
PROJECT = candidates[0].parent
for required in ("requirements.txt", "test_lab.py", "test_capacity.py", "test_v3.py",
                 "v3_protocol.json", "v3_protocol.sha256", "shadow_replay.py", "evaluate_v3.py"):
    if not (PROJECT / required).is_file():
        raise ValueError(f"แพ็กเกจไม่ครบ: {required}")
print("พร้อมรัน:", PROJECT.name)

# Verify the packaged files without creating or updating a freeze.
import json
import hashlib
manifest_path = PROJECT / "package_manifest.json"
if not manifest_path.is_file():
    raise ValueError("ไม่พบรายการตรวจสอบแพ็กเกจ")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("version") != "0.3.0":
    raise ValueError("รายการตรวจสอบไม่ใช่แพ็กเกจ v0.3.0")
for entry in manifest["contents"]:
    path = (PROJECT / entry["path"]).resolve()
    if not path.is_relative_to(PROJECT.resolve()) or not path.is_file():
        raise ValueError("เส้นทางในรายการตรวจสอบไม่ถูกต้อง")
    if path.stat().st_size != entry["bytes"]:
        raise ValueError(f"ขนาดไฟล์ไม่ตรง: {entry['path']}")
    if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
        raise ValueError(f"Hash ไม่ตรง: {entry['path']}")
print("ตรวจไฟล์ในแพ็กเกจแล้ว:", len(manifest["contents"]))


## 2. ติดตั้งและตรวจโค้ด

ติดตั้งตาม requirements ของแพ็กเกจ จากนั้นตรวจต้นแบบเดิม, v0.2 และ v0.3 พร้อม self-test ของ shadow adapter หากตรวจไม่ผ่าน เซลล์จะหยุดพร้อมรายละเอียด

ตรวจ frozen protocol ด้วย `verify_frozen()` ก่อนรัน tests โดยไม่สร้างหรือแก้ protocol ค่า hash ของโค้ดและสภาพแวดล้อมต้องตรงกับแพ็กเกจ

เซลล์นี้บันทึกจำนวน tests, failures, errors และสถานะที่เกิดขึ้นจริงใน session ปัจจุบันเป็น JSON นอกโฟลเดอร์โครงการ แล้วใช้หลักฐานใหม่นี้ประเมินเกณฑ์ advancement หลังรัน validation ไม่คัดลอกผลทดสอบเดิมมานับแทน

In [ ]:
import subprocess
import sys

def run_python(*arguments):
    command = [sys.executable, *map(str, arguments)]
    print("กำลังรัน:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

run_python("-m", "pip", "install", "-r", PROJECT / "requirements.txt")
run_python("-c", "from freeze_v3 import verify_frozen; print(verify_frozen()['protocol_sha256'])")

# Store fresh test evidence outside the verified project/package tree.
TEST_EVIDENCE = EXTRACT_ROOT / "session_unit_tests.json"
if TEST_EVIDENCE.resolve().is_relative_to(PROJECT.resolve()):
    TEST_EVIDENCE = Path(tempfile.mkdtemp(prefix="ans_session_tests_")) / "session_unit_tests.json"

unit_test_script = r"""
import json
import platform
import sys
import unittest
from pathlib import Path
import numpy as np

modules = ["test_lab", "test_capacity", "test_v3"]
suite = unittest.defaultTestLoader.loadTestsFromNames(modules)
result = unittest.TextTestRunner(verbosity=2).run(suite)
evidence = {
    "modules": modules,
    "tests_run": result.testsRun,
    "failures": len(result.failures),
    "errors": len(result.errors),
    "skipped": len(result.skipped),
    "expected_failures": len(result.expectedFailures),
    "unexpected_successes": len(result.unexpectedSuccesses),
    "successful": result.wasSuccessful(),
    "python": platform.python_version(),
    "numpy": np.__version__,
}
Path(sys.argv[1]).write_text(json.dumps(evidence, indent=2) + "\n", encoding="utf-8")
raise SystemExit(0 if result.wasSuccessful() else 1)
"""
run_python("-c", unit_test_script, TEST_EVIDENCE)
print("บันทึกผล tests ที่รันใน session นี้แล้ว:", TEST_EVIDENCE.name)
run_python("shadow_replay.py", "--self-test")


## 3. รัน validation ชุดใหม่

แต่ละสถานการณ์มีช่วงประเมินเดิม 60 หรือ 96 นาที ตามด้วยช่วงติดตาม 32 นาทีที่ไม่มีงานเข้าใหม่และคงกำลังการผลิตจริงแถวสุดท้าย เงื่อนไขช่วงติดตามนี้เป็นสมมติฐานจำลอง ไม่ได้บอกอนาคตให้ตัวควบคุม

ทุกครั้งที่รันเซลล์นี้จะใช้โฟลเดอร์ใหม่ การรันซ้ำตรวจความทำซ้ำได้ ไม่เพิ่มหลักฐานอิสระทางสถิติ

หลัง workflow เสร็จ จะเรียก evaluator ตามเกณฑ์ที่ตรึงไว้ พร้อมผล tests ของ session นี้ และสร้าง decision.json โดยอัตโนมัติ สถานะ workflow สำเร็จกับผล advancement ผ่าน/ไม่ผ่านเป็นคนละข้อสรุป

In [ ]:
from datetime import datetime, timezone
import uuid

run_name = datetime.now(timezone.utc).strftime("validation_%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUTPUT_ROOT = Path("/content/ans_v030_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = OUTPUT_ROOT / run_name
if RESULTS.exists():
    raise FileExistsError("ต้องใช้โฟลเดอร์ใหม่สำหรับการทดลอง")
if not TEST_EVIDENCE.is_file():
    raise RuntimeError("กรุณารันเซลล์ tests เพื่อสร้างหลักฐานใหม่ของ session นี้")
run_python("run_v3.py", "--split", "validation", "--out", RESULTS)
for filename in ("summary.json", "episodes.csv", "verification.json"):
    if not (RESULTS / filename).is_file():
        raise RuntimeError(f"ไม่พบผลลัพธ์: {filename}")

# Evaluate the exact frozen acceptance rule with this session's actual tests.
run_python("evaluate_v3.py", "--out", RESULTS, "--tests", TEST_EVIDENCE,
           "--report", RESULTS / "decision.json")
decision = json.loads((RESULTS / "decision.json").read_text(encoding="utf-8"))
verification = json.loads((RESULTS / "verification.json").read_text(encoding="utf-8"))
print("การประมวลผล workflow:", verification["status"])
print("เกณฑ์หลัก:", "PASS" if decision["primary"]["passed"] else "FAIL")
print("Noninferiority เทียบ v0.2:", "PASS" if decision["noninferiority"]["passed"] else "FAIL")
print("เกณฑ์ advancement รวม:", "PASS" if decision["advancement_passed"] else "FAIL")
if not decision["advancement_passed"]:
    print("การรันสำเร็จไม่ได้แปลว่าตัวเลือกผ่านเป้าหมาย:", ", ".join(decision["failed_criteria"]))
print("ผลและ decision ของการรันนี้:", run_name)


## 4. อ่านผลแต่ละเงื่อนไขแยกกัน

Mean backlog, terminal backlog, moves และ resource-minutes lost ในตารางแรกนับเฉพาะช่วงประเมินเดิม ส่วนตารางติดตามแสดงงานค้างท้าย 32 นาที, เวลาระบายที่จำกัดไว้ 32 นาที และสัดส่วนที่ยังระบายไม่หมด

ค่าระยะเวลาที่จำกัดไว้ต้องอ่านคู่กับสัดส่วนที่ยังไม่หมดคิว ไม่ใช่เวลาระบายจริงที่ทราบครบทุกกรณี Resource-minutes lost เป็นทรัพยากรที่ใช้ทำงานไม่ได้ระหว่างขนย้าย ไม่ใช่จำนวนเงิน

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

summary = json.loads((RESULTS / "summary.json").read_text(encoding="utf-8"))
policies = ["static", "abc_capacity", "capacity_v2", "capacity_balance", "v3_signal_only", "v3_full"]
names = {
    "static":"Static", "abc_capacity":"ABC + capacity gate",
    "capacity_v2":"Capacity v0.2", "capacity_balance":"Observed-rate balance",
    "v3_signal_only":"v0.3 signal only", "v3_full":"v0.3 full",
}
regime_names = {"instant":"ย้ายทันที", "transit_1":"ขนย้าย 1 นาที", "transit_3":"ขนย้าย 3 นาที"}
print("Scenarios:", summary["scenarios"], "| Scenario/regime pairs:", summary["scenario_regime_pairs"],
      "| Episodes:", summary["episodes"])
print("Original minutes:", summary["original_minutes"], "| Follow-up minutes:", summary["followup_minutes"])

for regime, label in regime_names.items():
    result = summary["regimes"][regime]
    overall = pd.DataFrame.from_dict(result["overall"], orient="index").reindex(policies)
    print("\n", label)
    primary_columns = ["mean_backlog", "end_backlog", "moves", "resource_minutes_lost"]
    display(overall[primary_columns].rename(index=names, columns={
        "mean_backlog":"Mean backlog (original H)",
        "end_backlog":"Backlog at original H",
        "moves":"Moves (original H)",
        "resource_minutes_lost":"Resource-minutes lost (original H)",
    }).round(3))
    followup = overall[["tail_auc", "tail_end_backlog", "drain_time_capped", "drain_uncleared_fraction"]].copy()
    followup["drain_uncleared_fraction"] *= 100
    display(followup.rename(index=names, columns={
        "tail_auc":"Tail backlog AUC",
        "tail_end_backlog":"Backlog after follow-up",
        "drain_time_capped":"Drain time capped at 32 min",
        "drain_uncleared_fraction":"Still uncleared (%)",
    }).round(3))
    fig, axis = plt.subplots(figsize=(10, 3.7), constrained_layout=True)
    axis.barh([names[p] for p in policies], overall["mean_backlog"],
              color=["#8293A6","#8293A6","#8293A6","#4D7EB5","#4D7EB5","#15856B"])
    axis.invert_yaxis()
    axis.set_title(f"{regime}: original-horizon mean backlog")
    axis.set_xlabel("Work units; lower is better")
    axis.grid(axis="x", alpha=.2)
    axis.set_axisbelow(True)
    plt.show()

## 5. ดูผลดีเฉลี่ยควบคู่กับกรณีที่แย่ลง

ตารางแรกของแต่ละเงื่อนไขแสดงผล V3 full เทียบกับวิธีอื่น ค่า reduction เป็นบวกหมายถึงงานค้างลดลง ช่วง 95% เป็น paired bootstrap ของสถานการณ์ภายในครอบครัวที่กำหนดไว้

ตารางความเสี่ยงคำนวณ **แต่ละวิธีเทียบกับ Static**: harm คือส่วนที่งานค้างเฉลี่ยมากกว่า Static โดยไม่นับค่าติดลบ CVaR90 คือค่าเฉลี่ยของ harm ในกลุ่มแย่ที่สุด 10% ของกรณี ไม่ใช่ความน่าจะเป็นที่ระบบจะปลอดภัย

In [ ]:
for regime, label in regime_names.items():
    result = summary["regimes"][regime]
    print("\n", label, "- paired comparisons")
    rows = []
    for baseline, comparison in result["comparisons"]["v3_full"].items():
        ci = comparison["reduction_pct_ci95"]
        rows.append({
            "Baseline":names.get(baseline, baseline),
            "Mean backlog reduction (%)":comparison["reduction_pct"],
            "95% CI low":ci[0] if ci else None,
            "95% CI high":ci[1] if ci else None,
            "Wins":comparison["wins"], "Ties":comparison["ties"], "Losses":comparison["losses"],
        })
    display(pd.DataFrame(rows).round(3))
    risk = pd.DataFrame.from_dict(result["static_relative_harm"], orient="index").reindex(policies)
    risk["frequency"] *= 100
    display(risk[["frequency","mean_positive_harm","cvar90","maximum_harm"]].rename(
        index=names, columns={
            "frequency":"Cases worse than Static (%)",
            "mean_positive_harm":"Mean positive harm vs Static",
            "cvar90":"Worst-10% mean harm vs Static",
            "maximum_harm":"Maximum harm vs Static",
        }).round(3))

decision_path = RESULTS / "decision.json"
if not decision_path.is_file():
    raise RuntimeError("ยังไม่มี decision ของการรันนี้ กรุณารันเซลล์ validation และ evaluator ให้ครบ")
decision = json.loads(decision_path.read_text(encoding="utf-8"))
print("\nWorkflow execution:", json.loads((RESULTS / "verification.json").read_text())["status"])
print("Advancement:", "PASS" if decision["advancement_passed"] else "FAIL")
decision_rows = [
    {"Criterion":"primary", "Result":"PASS" if decision["primary"]["passed"] else "FAIL", "Failed regimes":""},
    {"Criterion":"noninferiority", "Result":"PASS" if decision["noninferiority"]["passed"] else "FAIL", "Failed regimes":""},
]
for name, result in decision["advancement_checks"].items():
    decision_rows.append({
        "Criterion":name,
        "Result":"PASS" if result["passed"] else "FAIL",
        "Failed regimes":", ".join(result.get("failed_regimes", [])),
    })
display(pd.DataFrame(decision_rows))
print("การรันสำเร็จไม่ได้ยืนยันว่าเกณฑ์ advancement ผ่าน ต้องอ่านผลแต่ละข้อข้างบน")
print("ช่วงความเชื่อมั่นเป็นความแปรผันในแบบจำลองสังเคราะห์นี้ การเปรียบเทียบรองไม่ได้ปรับสำหรับหลายการเปรียบเทียบ")


## 6. ตัวอย่าง shadow replay ที่อ่านอย่างเดียว

เซลล์นี้อ่านตัวอย่างสังเคราะห์ที่บันทึกไว้แล้ว เพื่อแสดงคำแนะนำบนเส้นทางจริงของตัวอย่าง ข้อเสนอทั้งหมด **ไม่ถูกนำไปใช้** และไม่ถูกนับว่าเป็นการทดลองนโยบายเพิ่ม

ถ้าเห็นข้อเสนอซ้ำในนาทีติดกันเป็นเรื่องที่คาดได้ เพราะการไม่ทำตามข้อเสนอก่อนไม่ควรเลื่อนนาฬิกา dwell ข้อเสนอเหล่านี้ไม่ใช่แผนที่สามารถปฏิบัติเรียงกันได้ทันที และ replay นี้ยังบอกไม่ได้ว่าจะลดงานค้างจริงเท่าไร

In [ ]:
SHADOW_OUTPUT = RESULTS / ("shadow_recommendations_" + uuid.uuid4().hex[:8] + ".json")
run_python("shadow_replay.py", "--input", PROJECT / "synthetic_example.json", "--out", SHADOW_OUTPUT)
shadow = json.loads(SHADOW_OUTPUT.read_text(encoding="utf-8"))
print("Source:", shadow["source_kind"], "| Recommendations applied:", False)
print("Recommended moves:", shadow["recommended_move_count"],
      "| Recorded actual moves:", shadow["recorded_actual_move_count"])
selected = [record for record in shadow["recommendations"]
            if any(abs(x)>1e-6 for x in record["recommended_delta"])]
display(pd.DataFrame(selected)[["t","recommended_delta","reason","recorded_actual_delta",
                               "last_actual_change_after"]].head(15))

## 7. ดาวน์โหลดผลที่รันใหม่

ZIP นี้รวมผล validation รายแถว, สรุป, หลักฐานการตรวจ และตัวอย่าง shadow replay ที่เพิ่งรัน เก็บคู่กับ ZIP ต้นฉบับเพื่อใช้ตรวจสอบหรือทำซ้ำ

In [ ]:
import shutil

# Keep the actual fresh test evidence with the decision that used it.
shutil.copy2(TEST_EVIDENCE, RESULTS / "session_unit_tests.json")

results_archive = Path(shutil.make_archive(
    str(OUTPUT_ROOT / (run_name + "_results")), "zip", root_dir=RESULTS
))
print("ดาวน์โหลด:", results_archive.name)
files.download(str(results_archive))